<h1> Libraries </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

import outlines #https://dottxt-ai.github.io/outlines/latest/
from pydantic import BaseModel
from typing import List

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)

<h1> Criação de dataset sem técnicas com o modelo Microsoft/Phi-4 </h1>

In [ ]:
modelo = AutoModelForCausalLM.from_pretrained (r"microsoft/phi-4", quantization_config = quant, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (r"microsoft/phi-4")

#modelo.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")
#tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")

In [ ]:
x = 0
y = 15

modelo.eval()
for i in range (len(documentos) // 15):


    prompt = f"""
    <|im_start|>system<|im_sep|>

    És um assistente de IA com a função de Senior Data Scientist que tens como único e principal objetivo 
    criar um dataset sintético de acordo com os dados fornecidos. Deves criar perguntas de acordo com o 
    contexto e selecionar quais são os chunks mais relevantes para responder a essas perguntas.

    <REGRAS>

    1. És um modelo que retorna outputs apenas em formato JSON. Qualquer coisa retornada num formato diferente está errada!
    2. Deves ter em conta e ser fiel aos dados fornecidos.
    3. Cria perguntas concisas e diretas de acordo com o contexto fornecido.
    4. Deves ter em conta o 'chunk_id' de cada contexto e selecionar quais os 'chunk_id' mais relevantes para responder à pergunta criada. Tenta sempre selecionar mais que um 'chunk_id'
    5. Por cada contexto fornecido retorna no máximo 5 pares de perguntas e 'chunk_id' ideal para responder à pergunta.

    </REGRAS>


    <EXEMPLO_CONTEXTO>

    'chunk_id': 1, 'title': 'ABC_Máquinas_Elétricas.pdf', 'content': 'Actualmente, podemos considerar as máquinas eléctricas (motores, geradores e transformadores)'

    </EXEMPLO_CONTEXTO>


    <EXEMPLO_OUTPUT>

        "query": "Que tipos de máquinas elétricas existem ?",
        "chunk_id": [1, 2]

    </EXEMPLO_OUTPUT> 

    Tens como principal e única função responder de acordo com o contexto fornecido e criar perguntas 
    de acordo com o contexto e selecionar os chunk_id mais relevantes para responder à pergunta
    criada. Lembra-te que é fulcral que o output seja em formato JSON.
    <|im_end|>


    <|im_start|>user<|im_sep|>

    {documentos[x:y]}

    <|im_end|>


    <|im_start|>assistant<|im_sep|>


    """

    tokens = tokenizer (prompt, return_tensors = "pt").to (device)

    with torch.no_grad():
        logits = modelo.generate (**tokens, max_new_tokens = 180, use_cache = True)

    input_length = tokens["input_ids"].shape[1]
    response_tokens = logits[0][input_length:]

    output = tokenizer.decode(response_tokens, skip_special_tokens = True)

    with open ("dataset-Phi4.json", "a", encoding = "utf-8") as f:
        f.write (output + "\n")

    x = y
    y = y + 15

    del tokens, logits, output
    #gc.collect()
    torch.cuda.empty_cache()

    print (f"{i} Done")


<h1> Criação de dataset com Constraint Decoding com o modelo Mistral-7B-Instruct </h1>

In [ ]:
path = r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3-Q4"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (path)

In [ ]:
###teste solto

prompt = "Olá, quem foi o primeiro rei de Portugal ?"

tokens = tokenizer (prompt, return_tensors = "pt").to (device)

with torch.no_grad():
    logits = model.generate (**tokens, max_new_tokens = 512)

input_length = tokens["input_ids"].shape[1]
response_tokens = logits[0][input_length:]

output = tokenizer.decode(response_tokens, skip_special_tokens = True)

In [ ]:
#definiçaõ de formato


class PARES (BaseModel):
    query: str
    chunk_id: List[int]

class DATASET (BaseModel):
    data: List[PARES]


model = outlines.from_transformers (model, tokenizer)

In [ ]:
x = 0
y = 15


for i in range (len(documentos) // 15 - 3):

    prompt = f"""
    <s>system

    És um assistente de IA com a função de Senior Data Scientist que tens como único e principal objetivo 
    criar um dataset sintético de acordo com os dados fornecidos. Deves criar perguntas de acordo com o 
    contexto e selecionar quais são os chunks mais relevantes para responder a essas perguntas. 

    <REGRAS>

    1. És um modelo que retorna outputs apenas em formato JSON. Qualquer coisa retornada num formato diferente está errada!
    2. Deves ter em conta e ser fiel aos dados fornecidos.
    3. Cria perguntas concisas e diretas de acordo com o contexto fornecido.
    4. Deves ter em conta o 'chunk_id' de cada contexto e selecionar quais os 'chunk_id' mais relevantes para responder à pergunta criada. Tens que selecionar sempre mais que um 'chunk_id'!!
    5. Não podes repetir perguntas e retorna apenas 5 exemplos por prompt!
    6. As perguntas têm de estar em Português Europeu!

    </REGRAS>

    Tens como principal e única função responder de acordo com o contexto fornecido e criar perguntas 
    de acordo com o contexto e selecionar os chunk_id mais relevantes para responder à pergunta
    criada. Lembra-te que é fulcral que o output seja em formato JSON e em Português Europeu.
    </s>

    
    <s>user

    <CONTEXTO>

    {documentos[x:y]}

    </CONTEXTO
    </s>

    <s>assistant

    """

    result = model (prompt, output_type = DATASET, max_new_tokens = 512) #temp, top_k, top_p
    #display (result)


    with open ("dataset-Mistral.json", "a", encoding = "utf-8") as f:
        f.write (result + "\n") 
    
    x = y
    y = y + 15

    torch.cuda.empty_cache()